Part A

In [22]:
import open3d as o3d
import copy
import numpy as np

# Initialize functions
def draw_registration_result(source, target):
    """
    param: source - source point cloud
    param: target - target point cloud
    param: transformation - 4 X 4 homogeneous transformation matrix
    """
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.paint_uniform_color([1, 0.706, 0])#Yellow
    target_temp.paint_uniform_color([0, 0.651, 0.929])#Blue
    o3d.visualization.draw_geometries([source_temp, target_temp], zoom=0.4459, front=[0.9288, -0.2951, -0.2242], lookat=[1.6784, 2.0612, 1.4451], up=[-0.3402, -0.9189, -0.1996])

def find_nearest_neighbors_indices(source_pc, target_pc, k):
    """Returns indices of nearest neighbors"""
    tree = o3d.geometry.KDTreeFlann(target_pc)
    indices_list = []
    
    for point in source_pc.points:
        [_, idx, _] = tree.search_knn_vector_3d(point, k)
        indices_list.append(idx[0])  # Stores just the index
    
    return np.asarray(indices_list)




In [25]:
def icp(source, target):
    """
    ICP that modifies source in place and returns estimated transformation
    """
    # ALL variables initialized INSIDE the function
    T_total = np.eye(4)
    prev_cost = float('inf') 
    threshold =  1e-4        
    max_iter = 50             
    
    target_points = np.asarray(target.points)
    source_working = copy.deepcopy(source)

    for i in range(max_iter):
        source_points = np.asarray(source_working.points)
        indices = find_nearest_neighbors_indices(source_working, target, 1)
        matched_target = target_points[indices]
        source_centroid = np.mean(source_points, axis=0)
        target_centroid = np.mean(matched_target, axis=0)
        source_centered = source_points - source_centroid
        target_centered = matched_target - target_centroid
        H = source_centered.T @ target_centered
        U, S, Vt = np.linalg.svd(H)
        R = Vt.T @ U.T
        if np.linalg.det(R) < 0:
            Vt[2, :] *= -1
            R = Vt.T @ U.T
        t = target_centroid - R @ source_centroid
        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = t
        source_working.transform(T)
        T_total = T @ T_total
        source_transformed = np.asarray(source_working.points)
        cost = np.mean(np.sum((source_transformed - matched_target)**2, axis=1))
        print(f"Iter {i}: cost = {cost:.6f}")
        if abs(prev_cost - cost) < threshold:
            print(f"Converged at iteration {i}")
            break
        prev_cost = cost

    source.transform(T_total)
    return T_total

In [26]:
# Demo Part A 
demo_icp_pcds = o3d.data.DemoICPPointClouds()
source = o3d.io.read_point_cloud(demo_icp_pcds.paths[0])
target = o3d.io.read_point_cloud(demo_icp_pcds.paths[1])

source_copy_a = copy.deepcopy(source)
part_a = icp(source_copy_a, target)


print(part_a)

Iter 0: cost = 0.123231
Iter 1: cost = 0.073290
Iter 2: cost = 0.059223
Iter 3: cost = 0.052204
Iter 4: cost = 0.048610
Iter 5: cost = 0.046973
Iter 6: cost = 0.046216
Iter 7: cost = 0.045796
Iter 8: cost = 0.045528
Iter 9: cost = 0.045340
Iter 10: cost = 0.045207
Iter 11: cost = 0.045110
Converged at iteration 11
[[ 0.83208122 -0.01863096 -0.55434081  1.37614674]
 [-0.17493527  0.93960959 -0.29416232  0.98971966]
 [ 0.52634447  0.3417407   0.77857228 -1.40455342]
 [ 0.          0.          0.          1.        ]]


Part B

In [27]:
# Part B - Cat mesh 
mesh = o3d.io.read_triangle_mesh("C:\\Users\\zeine\\Downloads\\cat1_un4 (1).obj")
mesh.compute_vertex_normals()
pcd = mesh.sample_points_uniformly(5000)

source_original = copy.deepcopy(pcd)
target_original = copy.deepcopy(pcd)

R_true = source_original.get_rotation_matrix_from_xyz((0.5, 0.2, 0.1))
t_true = np.array([0.3, -0.4, 0.2])
T_true = np.eye(4)
T_true[:3, :3] = R_true
T_true[:3, 3] = t_true
target_original.transform(T_true)

# === WINDOW 1: BEFORE ICP — two separate cats ===
print("WINDOW 1: BEFORE ICP - Two separate cats")
draw_registration_result(source_original, target_original)

# Run ICP on copies
source_copy = copy.deepcopy(source_original)
target_copy = copy.deepcopy(target_original)
T_est = icp(source_copy, target_copy)

# === WINDOW 2: AFTER ICP — source_copy is now aligned to target_copy ===

print("\nWINDOW 2: AFTER ICP - Cats aligned")
#Displays randomly whether blue or yellow cat as tehy are perfectly alligned 
draw_registration_result(source_copy, target_copy)
pts = np.asarray(source_copy.points)
bbox_size = pts.max(axis=0) - pts.min(axis=0)
small_shift = bbox_size[0] * 0.05  # only 5% shift cats look aligned but both colors show

source_window2 = copy.deepcopy(source_copy)
shifted_pts = np.asarray(source_window2.points).copy()
shifted_pts[:, 0] += small_shift
source_window2.points = o3d.utility.Vector3dVector(shifted_pts)

draw_registration_result(source_window2, target_copy)

# === WINDOW 3: shift source away from target to show them separately ===
print("\nWINDOW 3: Source shifted away from target")
pts = np.asarray(source_copy.points)
bbox_size = pts.max(axis=0) - pts.min(axis=0)
shift_amount = bbox_size[0] * 0.6 #60% shift 

source_shifted = copy.deepcopy(source_copy)
shifted_pts = np.asarray(source_shifted.points).copy()
shifted_pts[:, 0] += shift_amount
source_shifted.points = o3d.utility.Vector3dVector(shifted_pts)

draw_registration_result(source_shifted, target_copy)

# === Results ===
print("\nGround truth T:\n", T_true)
print("Estimated T:\n", T_est)

R_error = np.linalg.norm(T_true[:3,:3] - T_est[:3,:3])
t_error = np.linalg.norm(T_true[:3,3] - T_est[:3,3])

print(f"\n{'='*50}")
print(f"Rotation error:    {R_error:.4e}")
print(f"Translation error: {t_error:.4e}")
print(f"Total error:       {np.linalg.norm(T_true - T_est):.4e}")

if R_error < 0.01 and t_error < 0.1:
    print("\n Good alignment achieved!")
else:
    print("\n  Some error remains")

WINDOW 1: BEFORE ICP - Two separate cats
Iter 0: cost = 381.176238
Iter 1: cost = 157.430195
Iter 2: cost = 89.471217
Iter 3: cost = 63.017521
Iter 4: cost = 46.989860
Iter 5: cost = 36.032312
Iter 6: cost = 28.074360
Iter 7: cost = 21.866369
Iter 8: cost = 16.943851
Iter 9: cost = 13.078352
Iter 10: cost = 10.089986
Iter 11: cost = 7.993858
Iter 12: cost = 6.430647
Iter 13: cost = 5.241608
Iter 14: cost = 4.313666
Iter 15: cost = 3.631401
Iter 16: cost = 3.109107
Iter 17: cost = 2.693493
Iter 18: cost = 2.326302
Iter 19: cost = 1.984657
Iter 20: cost = 1.645381
Iter 21: cost = 1.247745
Iter 22: cost = 0.688231
Iter 23: cost = 0.115302
Iter 24: cost = 0.000399
Iter 25: cost = 0.000000
Iter 26: cost = 0.000000
Converged at iteration 26

WINDOW 2: AFTER ICP - Cats aligned

WINDOW 3: Source shifted away from target

Ground truth T:
 [[ 0.97517033 -0.0978434   0.19866933  0.3       ]
 [ 0.18238338  0.86368946 -0.46986895 -0.4       ]
 [-0.12561503  0.49443624  0.86008934  0.2       ]
 [ 0.